In [ ]:
import os


def load_clean_data(filepath="../data/raw/raw_usd_ngn.csv"):
    df = pd.read_csv(filepath, parse_dates=["Date"], index_col="Date")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    # Apply our data engineering fix from earlier
    df = df[df["Close"] > 100]
    df["Log_Returns"] = np.log(df["Close"] / df["Close"].shift(1))
    return df.dropna()


df = load_clean_data()

In [ ]:
import numpy as np
import pandas as pd
# result = detect_structural_breaks(df)


def build_feature_matrix(df, change_points):
    """
    Transforms clean time-series logs into a structured,
    supervised machine learning feature matrix.
    """
    matrix = df.copy()

    # Target: Tomorrow's log return (what the model tries to guess)
    matrix['Target'] = matrix['Log_Returns'].shift(-1)

    # Lag Features: Historical returns mapping market momentum
    for lag in [1, 2, 3, 5]:
        matrix[f'Lag_{lag}'] = matrix['Log_Returns'].shift(lag)

    # Volatility Windows: Tracking market panic & ARCH/GARCH clustering effects
    matrix['Vol_Weekly'] = matrix['Log_Returns'].rolling(window=5).std()
    matrix['Vol_Monthly'] = matrix['Log_Returns'].rolling(window=21).std()
    matrix['Realized_Var_Weekly'] = (
        matrix['Log_Returns'] ** 2).rolling(window=5).mean()

    # Regime Mapping: Assigning structural policy IDs based on change points
    matrix['Regime'] = 0
    for i, bkp in enumerate(change_points[:-1]):
        matrix.iloc[bkp:change_points[i+1],
                    matrix.columns.get_loc('Regime')] = i + 1

    # Drop edge rows containing NaNs generated by shifting, lagging, and rolling windows
    return matrix.dropna()


# Step A: Generate the unified feature dataframe
print("Building the feature matrix...")
df_features = build_feature_matrix(df, result)

# Step B: Separate independent variables (X) from the dependent target (y)
feature_cols = [
    'Lag_1', 'Lag_2', 'Lag_3', 'Lag_5',
    'Vol_Weekly', 'Vol_Monthly',
    'Realized_Var_Weekly',
    'Regime'
]

X = df_features[feature_cols]
y = df_features['Target']

# Step C: Chronological Time-Series Split (80% Train / 20% Validation)
# This prevents future data leakage into past market training cycles
split_idx = int(len(df_features) * 0.80)

X_train, X_val = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_val = y.iloc[:split_idx], y.iloc[split_idx:]


print("Feature Pipeline Execution: SUCCESS")
print("-" * 50)
print(f"Total Cleaned Datapoints  : {df_features.shape[0]}")
print(f"Training Features Set (X) : {X_train.shape}")
# Checks the validation set to ensure the model evaluates on the current economic era
print(f"Validation Features Set (X): {X_val.shape}")
print(
    f"Validation Era Baseline    : From {df_features.index[split_idx].strftime('%Y-%m-%d')} onward")
print("-" * 50)

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# 1. Initialize and train the baseline Linear Regression model
baseline_model = LinearRegression()
baseline_model.fit(X_train, y_train)

# 2. Predict tomorrow's log returns on the unseen validation set (post-April 2024)
y_pred = baseline_model.predict(X_val)

# 3. Calculate evaluation metrics
mae = mean_absolute_error(y_val, y_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

# 4. Calculate Directional Accuracy (Did it correctly guess an upward or downward move?)
correct_direction = np.sign(y_val.values) == np.sign(y_pred)
directional_accuracy = np.mean(correct_direction) * 100

# 5. Display the performance summary
print("=========================================")
print("      BASELINE MODEL EVALUATION         ")
print("=========================================")
print(f"Mean Absolute Error (MAE)  : {mae:.6f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.6f}")
print(f"Directional Accuracy       : {directional_accuracy:.2f}%")
print("=========================================\n")

# 6. Check feature weights to see what the baseline values most
print("CRITICAL FEATURE COEFFICIENTS:")
print("-" * 41)
for col, coef in zip(feature_cols, baseline_model.coef_):
    print(f"{col:<22} | Coefficient: {coef:.6f}")